In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

In [4]:
df1=pd.read_parquet('D:/AHU/rdm-wach-ai/paraquet_data/raw/raw_one_year.parquet',engine="pyarrow")
df1.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'D:/AHU/rdm-wach-ai/paraquet_data/raw/raw_one_year.parquet'

In [32]:
df1.columns

Index(['result', 'table', 'controller', 'site', 'power_factor_avg', 'units',
       'current_l1', 'current_l2', 'current_l3', 'volts_l1_n', 'volts_l2_n',
       'volts_l3_n', 'apparent_power_total', 'power_l1', 'power_l2',
       'power_l3', 'power_total'],
      dtype='object')

Apply the range

In [29]:
# ranges = {
#     'power_total': (0, 10.0),
#     'current_l1': (0, 20.0),
#     'current_l2': (0, 20.0),
#     'current_l3': (0, 20.0),
#     'volts_l1_n': (200, 260),
#     'volts_l2_n': (200, 260),
#     'volts_l3_n': (200, 260),
#     'power_factor_avg': (0, 1.0),
#     #'apparent_power_total': (0, 15.0)
# }
# for col, (min_val, max_val) in ranges.items():
#     if col in df1.columns:
#         df1.loc[(df1[col] < min_val) | (df1[col] > max_val), col] = np.nan
# # Filter rows where any column in your ranges is NaN
# invalid_rows = df1[df1[list(ranges.keys())].isna().any(axis=1)]
# print(invalid_rows)

# # Rows with any NaN in the selected columns
# num_invalid_rows = df1[list(ranges.keys())].isna().any(axis=1).sum()
# print("Number of rows with at least one out-of-range value:", num_invalid_rows)

# num_invalid_per_column = df1[list(ranges.keys())].isna().sum()
# print("Number of out-of-range values per column:\n", num_invalid_per_column)

# total_invalid = df1[list(ranges.keys())].isna().sum().sum()
# print("Total number of out-of-range values:", total_invalid)

# # Show first 10 rows with out-of-range values
# print(df1[df1[list(ranges.keys())].isna().any(axis=1)].head(10))



In [33]:
df1.describe()

,result,table,controller,site,power_factor_avg,units,current_l1,current_l2,current_l3,volts_l1_n,volts_l2_n,volts_l3_n,apparent_power_total,power_l1,power_l2,power_l3,power_total
count,9480.0,9480.000000,9480.0,9480.0,9480.000000,9480.0,9480.000000,9480.000000,9480.000000,9480.000000,9480.000000,9480.000000,9480.000000,9480.000000,9.480000e+03,9480.000000,9480.000000
mean,0.0,2.011339,0.0,0.0,0.785618,0.0,6.407143,5.566035,7.859544,235.793875,234.347287,236.220668,4.663905,1.282326,1.029363e+00,1.312494,3.623868
std,0.0,0.040573,0.0,0.0,0.077231,0.0,2.043525,1.638894,2.323497,2.060338,2.089668,2.129733,1.322923,0.444127,2.788383e-01,0.365058,0.990482
min,0.0,1.600000,0.0,0.0,0.486667,0.0,0.030000,0.030000,0.020000,157.066667,156.000000,157.333333,0.019876,0.007907,1.090450e-305,0.005387,0.019093
25%,0.0,2.000000,0.0,0.0,0.730000,0.0,5.610000,4.460000,6.133214,234.400000,233.083333,234.866667,3.936380,1.092450,9.317357e-01,1.141230,3.401037
50%,0.0,2.000000,0.0,0.0,0.780000,0.0,6.091429,5.663750,8.802679,235.875000,234.371429,236.300000,4.960071,1.398993,9.929310e-01,1.367576,3.555363
75%,0.0,2.000000,0.0,0.0,0.890000,0.0,8.210000,6.874286,9.140000,237.200000,235.900000,237.700000,5.693171,1.638129,1.259330e+00,1.501127,4.431110
max,0.0,4.000000,0.0,0.0,0.960000,0.0,10.812000,7.195714,10.922000,240.700000,239.625000,242.100000,6.680440,2.149984,1.335069e+00,1.884160,5.148878


flatline metrics

In [52]:
import pandas as pd
import numpy as np

def detect_constant(series, window=8):
    """
    Detect true constant sequences in a series.
    Returns a boolean mask where True = constant values.
    """
    # Step 1: mark changes
    change = series != series.shift()
    
    # Step 2: assign groups of consecutive same values
    group = change.cumsum()
    
    # Step 3: count size of each group
    group_size = group.map(group.value_counts())
    
    # Step 4: mark flatline if group size >= window
    constant_mask = group_size >= window
    
    return constant_mask
# Columns to check
flatline_metrics = [
    'power_total', 'current_l1', 'current_l2', 'current_l3',
    'volts_l1_n', 'volts_l2_n', 'volts_l3_n', 'power_factor_avg'
]

constant_masks = {}

for col in flatline_metrics:
    if col in df1.columns:
        mask = detect_constant(df1[col], window=8)
        constant_masks[col] = mask
        # Optional: add a flag column
        df1[f'{col}_constant_flag'] = mask.astype(int)

# Show how many points per column are constant
for col, mask in constant_masks.items():
    total_points = mask.sum()
    print(f"{col}: {total_points} constant points detected")


power_total: 0 constant points detected
current_l1: 0 constant points detected
current_l2: 0 constant points detected
current_l3: 0 constant points detected
volts_l1_n: 0 constant points detected
volts_l2_n: 0 constant points detected
volts_l3_n: 0 constant points detected
power_factor_avg: 24 constant points detected


In [54]:
periods = []

for col, mask in constant_masks.items():
    group = (mask != mask.shift()).cumsum()
    for g in group[mask].unique():
        period = df1.loc[group == g]
        periods.append({
            "column": col,
            "start_time": period.index[0],
            "end_time": period.index[-1],
            "duration": len(period),
            "value": period[col].iloc[0]
        })

periods_df = pd.DataFrame(periods)
print(periods_df.sort_values(["column", "start_time"]))


             column                start_time                  end_time  \
0  power_factor_avg 2025-11-05 12:45:00+00:00 2025-11-05 14:30:00+00:00   
1  power_factor_avg 2025-12-13 04:45:00+00:00 2025-12-13 06:30:00+00:00   
2  power_factor_avg 2025-12-15 02:00:00+00:00 2025-12-15 03:45:00+00:00   

   duration     value  
0         8  0.892857  
1         8  0.780000  
2         8  0.780000  


In [55]:
for col, mask in constant_masks.items():
    df1.loc[mask, col] = np.nan

# Interpolate only the points that were constant
if pd.api.types.is_datetime64_any_dtype(df1.index):
    df1[flatline_metrics] = df1[flatline_metrics].interpolate(method='time')
else:
    df1[flatline_metrics] = df1[flatline_metrics].interpolate(method='linear')


Gap Detection and Handling 

In [35]:

# List of features you want to handle
feature_columns = [
    'power_total',
    'current_l1',
    'current_l2',
    'current_l3',
    'volts_l1_n',
    'volts_l2_n',
    'volts_l3_n',
    'power_factor_avg'
]

# Loop over each feature
for col in feature_columns:
    #  Identify NaNs
    is_nan = df1[col].isna()
    
    #  Check if previous and next values exist
    prev_valid = ~df1[col].shift(1).isna()
    next_valid = ~df1[col].shift(-1).isna()
    
    #  Identify single gaps only
    single_gap = is_nan & prev_valid & next_valid
    
    #  Interpolate single gaps
    df1.loc[single_gap, col] = df1[col].interpolate(method='linear', limit=1)
    
    #  Extended gaps (≥2 consecutive intervals) remain NaN
    # No action required; they are left as NaN
    
# Count NaNs before interpolation
print("NaNs before handling:")
print(df1[feature_columns].isna().sum())

# Run your interpolation code here (already done)

# Count NaNs after interpolation
print("\nNaNs after handling:")
print(df1[feature_columns].isna().sum())


NaNs before handling:
power_total         0
current_l1          0
current_l2          0
current_l3          0
volts_l1_n          0
volts_l2_n          0
volts_l3_n          0
power_factor_avg    0
dtype: int64

NaNs after handling:
power_total         0
current_l1          0
current_l2          0
current_l3          0
volts_l1_n          0
volts_l2_n          0
volts_l3_n          0
power_factor_avg    0
dtype: int64


Feature completeness rule(null value checked)

In [ ]:

# Your feature columns
feature_columns = [
    'power_total',
    'current_l1',
    'current_l2',
    'current_l3',
    'volts_l1_n',
    'volts_l2_n',
    'volts_l3_n',
    'power_factor_avg'
]

#  Target Variable Rule ---
# Drop rows where target 'kw' is NaN
df = df1[df1['power_total'].notna()]

# Count NaNs per row for selected features
nan_count = df1[feature_columns].isna().sum(axis=1)

# Calculate the ratio of missing features per row
nan_ratio = nan_count / len(feature_columns)

# Keep rows where ≤50% features are missing
df1 = df1[nan_ratio <= 0.5]

# Optional: reset index
df1 = df1.reset_index(drop=True)

# Check missing values per column before cleaning
print("Missing values per column BEFORE cleaning:")
print(df1.isna().sum())

# Total missing values
print("\nTotal missing values in dataset:", df1.isna().sum().sum())

# Optional: Percentage of missing per column
print("\nPercentage of missing values per column:")
print((df1.isna().sum() / len(df1) * 100).round(2))

# Show rows with any missing values
missing_rows = df1[df1.isna().any(axis=1)]
print("Rows with missing values BEFORE cleaning:", len(missing_rows))
print(missing_rows.head(10))

# --- Target Variable Rule ---
df1 = df1[df1['power_total'].notna()]

# --- Feature Completeness Rule ---
feature_columns = [
    'power_total', 'current_l1','current_l2','current_l3',
    'volts_l1_n','volts_l2_n','volts_l3_n','power_factor_avg'
]

nan_count = df1[feature_columns].isna().sum(axis=1)
nan_ratio = nan_count / len(feature_columns)
df1 = df1[nan_ratio <= 0.5]

# Reset index
df1 = df1.reset_index(drop=True)



Missing values per column BEFORE cleaning:
result                  0
table                   0
controller              0
site                    0
power_factor_avg        0
units                   0
current_l1              0
current_l2              0
current_l3              0
volts_l1_n              0
volts_l2_n              0
volts_l3_n              0
apparent_power_total    0
power_l1                0
power_l2                0
power_l3                0
power_total             0
dtype: int64

Total missing values in dataset: 0

Percentage of missing values per column:
result                  0.0
table                   0.0
controller              0.0
site                    0.0
power_factor_avg        0.0
units                   0.0
current_l1              0.0
current_l2              0.0
current_l3              0.0
volts_l1_n              0.0
volts_l2_n              0.0
volts_l3_n              0.0
apparent_power_total    0.0
power_l1                0.0
power_l2                0.0
powe

In [36]:
print("\nMissing values per column AFTER cleaning:")
print(df1.isna().sum())

print("\nTotal missing values in dataset after cleaning:", df1.isna().sum().sum())

# Optional: show first few rows to see cleaned dataset
print("\nFirst 10 rows after cleaning:")
print(df1.head(10))



Missing values per column AFTER cleaning:
result                  0
table                   0
controller              0
site                    0
power_factor_avg        0
units                   0
current_l1              0
current_l2              0
current_l3              0
volts_l1_n              0
volts_l2_n              0
volts_l3_n              0
apparent_power_total    0
power_l1                0
power_l2                0
power_l3                0
power_total             0
dtype: int64

Total missing values in dataset after cleaning: 0

First 10 rows after cleaning:
                           result     table  controller  site  \
time                                                            
2025-11-05 06:00:00+00:00     0.0  1.961538         0.0   0.0   
2025-11-05 06:15:00+00:00     0.0  2.078947         0.0   0.0   
2025-11-05 06:30:00+00:00     0.0  2.000000         0.0   0.0   
2025-11-05 06:45:00+00:00     0.0  2.023810         0.0   0.0   
2025-11-05 07:00:00+00:00     

In [74]:
print("\nDataset info:")
print(df1.info())

print("\nDescriptive statistics:")
print(df1.describe())



Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9480 entries, 0 to 9479
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   result                9480 non-null   float64
 1   table                 9480 non-null   float64
 2   controller            9480 non-null   float64
 3   site                  9480 non-null   float64
 4   power_factor_avg      9480 non-null   float64
 5   units                 9480 non-null   float64
 6   current_l1            9480 non-null   float64
 7   current_l2            9480 non-null   float64
 8   current_l3            9480 non-null   float64
 9   volts_l1_n            9480 non-null   float64
 10  volts_l2_n            9480 non-null   float64
 11  volts_l3_n            9480 non-null   float64
 12  apparent_power_total  9480 non-null   float64
 13  power_l1              9480 non-null   float64
 14  power_l2              9480 non-null   float64
 15  power_

In [75]:
# I want time as columns not as index 
df1 = df1.reset_index()  # moves the index into a regular column

Gold dataset

In [76]:
total_rows = len(df1)
# Example: detect flatline for power_total
flatline_mask = detect_flatline(df1['power_total'], window=8)
flatline_drop_count = flatline_mask.sum()

# Drop flatline rows
df1 = df1[~flatline_mask]

target_nan_count = (~df1['power_total'].notna()).sum()  # before dropping
df1 = df1[df1['power_total'].notna()]  # drop rows

feature_columns = ['power_total','current_l1','current_l2','current_l3',
                   'volts_l1_n','volts_l2_n','volts_l3_n','power_factor_avg']
nan_count = df1[feature_columns].isna().sum(axis=1)
nan_ratio = nan_count / len(feature_columns)
sparse_row_count = (nan_ratio > 0.5).sum()


# Drop sparse rows
df1 = df1[nan_ratio <= 0.5]

gold_df = df1.copy()  # final cleaned dataset
gold_metrics = {
    'total_rows': total_rows,
    'gold_rows': len(gold_df),
    'gold_percentage': len(gold_df) / total_rows * 100,
    'flatline_rows_dropped': int(flatline_drop_count),
    'target_missing_rows_dropped': int(target_nan_count),
    'feature_sparse_rows_dropped': int(sparse_row_count),
}
print("Data cleaning summary metrics:")
for k, v in gold_metrics.items():
    print(f"{k}: {v}")


Data cleaning summary metrics:
total_rows: 9480
gold_rows: 8248
gold_percentage: 87.0042194092827
flatline_rows_dropped: 1232
target_missing_rows_dropped: 0
feature_sparse_rows_dropped: 0


Timeseries slot

In [77]:
numeric_cols = df1.select_dtypes(include='number').columns
df_numeric = df1[numeric_cols]
df_numeric

,index,result,table,controller,site,power_factor_avg,units,current_l1,current_l2,current_l3,volts_l1_n,volts_l2_n,volts_l3_n,apparent_power_total,power_l1,power_l2,power_l3,power_total
0,0,0.0,1.961538,0.0,0.0,0.79,0.0,0.6640,0.5500,0.710000,232.340000,230.860000,232.360000,0.446164,0.135066,0.100593,0.116841,0.352846
1,1,0.0,2.078947,0.0,0.0,0.79,0.0,0.6700,0.5500,0.718571,233.685714,232.185714,233.728571,0.450998,0.136399,0.101432,0.117955,0.355975
2,2,0.0,2.000000,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,233.928571,232.442857,234.028571,0.451970,0.136216,0.101376,0.118240,0.356162
3,3,0.0,2.023810,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,234.075000,232.575000,234.087500,0.452858,0.136388,0.101520,0.118302,0.355818
4,4,0.0,2.000000,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,234.425000,232.975000,234.487500,0.453960,0.136882,0.101749,0.118393,0.357117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9475,9475,0.0,2.000000,0.0,0.0,0.73,0.0,5.9400,5.6200,10.600000,236.500000,234.800000,236.700000,5.233240,1.058030,0.932165,1.845840,3.838310
9476,9476,0.0,1.952381,0.0,0.0,0.73,0.0,5.9400,5.6125,10.600000,236.500000,234.725000,236.625000,5.233668,1.059732,0.927292,1.843703,3.831455
9477,9477,0.0,2.000000,0.0,0.0,0.73,0.0,5.9400,5.6200,10.610000,236.400000,234.900000,236.700000,5.231420,1.063250,0.933172,1.842850,3.836670
9478,9478,0.0,2.000000,0.0,0.0,0.78,0.0,10.0100,5.6200,10.490000,234.200000,233.200000,234.500000,6.095990,1.993200,0.925165,1.815090,4.734360


In [87]:
#drop unwanted columns
drop_cols = ['result', 'controller', 'units','table','site','power_l1','power_l2','power_l3',
'apparent_power_total'
]
df_corr = df1.drop(columns=drop_cols, errors='ignore')
df=df_corr.copy()

Feature Engineering

In [88]:
df=df_corr.copy()
df.index=pd.to_datetime(df.index)

df = df.reset_index()

df["hour"] = df["time"].dt.hour
df["dayofweek"] = df["time"].dt.dayofweek
df["month"] = df["time"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)


In [89]:
df.columns

Index(['time', 'power_factor_avg', 'current_l1', 'current_l2', 'current_l3',
       'volts_l1_n', 'volts_l2_n', 'volts_l3_n', 'power_total', 'hour',
       'dayofweek', 'month', 'is_weekend'],
      dtype='object')

Lag Features

In [91]:
#lag features (15-min interval)  # 1 step, 1 hour, 1 day lag
for lag in [1,4,96]:
    df[f'lag_{lag}']=df['power_total'].shift(lag)

Rolling Statistics

In [92]:
df["rolling_mean_4"] = df["power_total"].shift(1).rolling(4).mean()
df["rolling_std_4"] = df["power_total"].shift(1).rolling(4).std()

In [ ]:
#Drop Nans after lagging
df=df.dropna()
df=df.drop(columns=['time'])

In [95]:
#df=df.drop(columns=['index'])
df.columns

Index(['power_factor_avg', 'current_l1', 'current_l2', 'current_l3',
       'volts_l1_n', 'volts_l2_n', 'volts_l3_n', 'power_total', 'hour',
       'dayofweek', 'month', 'is_weekend', 'lag_1', 'lag_4', 'lag_96',
       'rolling_mean_4', 'rolling_std_4'],
      dtype='object')

In [96]:
target_col="power_total"
feature_cols=df.drop(columns=["power_total"]).columns

In [2]:
feature_cols.columns

NameError: name 'feature_cols' is not defined

Rolling window

In [97]:
# 4. Rolling window parameters
rows_per_day = 96      # 15-minute intervals
train_days = 60
test_days = 7
train_size = train_days * rows_per_day
test_size = test_days * rows_per_day

r2_scores = []

In [98]:
# Rolling window parameters
rows_per_day = 96
train_days = 60
test_days = 7

train_size = train_days * rows_per_day
test_size = test_days * rows_per_day

# Metric storage
mae_scores = []
rmse_scores = []
mape_scores = []
r2_scores = []

for start in range(0, len(df) - train_size - test_size + 1, test_size):

    train_idx = slice(start, start + train_size)
    test_idx = slice(start + train_size, start + train_size + test_size)

    X_train = df.iloc[train_idx][feature_cols]
    y_train = df.iloc[train_idx][target_col]

    X_test = df.iloc[test_idx][feature_cols]
    y_test = df.iloc[test_idx][target_col]

    model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        objective='reg:squarederror',
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # ---- METRICS ----
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Safe MAPE (avoid divide by zero)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

    # Store
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    mape_scores.append(mape)
print("\nRolling Window Backtesting Results")
print("-"*50)
print(f"Number of splits: {len(r2_scores)}")

print("\nAverage Metrics Across All Splits")
print(f"Mean MAE  : {np.mean(mae_scores):.4f}")
print(f"Mean RMSE : {np.mean(rmse_scores):.4f}")
print(f"Mean MAPE : {np.mean(mape_scores):.2f}%")
print(f"Mean R²   : {np.mean(r2_scores):.4f}")

print("\nStandard Deviation of Metrics")
print(f"Std MAE   : {np.std(mae_scores):.4f}")
print(f"Std RMSE  : {np.std(rmse_scores):.4f}")
print(f"Std MAPE  : {np.std(mape_scores):.2f}%")
print(f"Std R²    : {np.std(r2_scores):.4f}")
print("\nSplit-wise Results")
print("-"*50)

for i in range(len(r2_scores)):
    print(f"Split {i+1}: "
          f"MAE={mae_scores[i]:.4f}, "
          f"RMSE={rmse_scores[i]:.4f}, "
          f"MAPE={mape_scores[i]:.2f}%, "
          f"R²={r2_scores[i]:.4f}")



Rolling Window Backtesting Results
--------------------------------------------------
Number of splits: 5

Average Metrics Across All Splits
Mean MAE  : 0.1604
Mean RMSE : 0.2135
Mean MAPE : 4.57%
Mean R²   : 0.4876

Standard Deviation of Metrics
Std MAE   : 0.1436
Std RMSE  : 0.1683
Std MAPE  : 4.02%
Std R²    : 0.8441

Split-wise Results
--------------------------------------------------
Split 1: MAE=0.4314, RMSE=0.5270, MAPE=12.18%, R²=-1.1965
Split 2: MAE=0.1747, RMSE=0.2448, MAPE=4.73%, R²=0.8178
Split 3: MAE=0.0957, RMSE=0.1325, MAPE=3.21%, R²=0.8765
Split 4: MAE=0.0297, RMSE=0.0591, MAPE=0.90%, R²=0.9715
Split 5: MAE=0.0703, RMSE=0.1041, MAPE=1.81%, R²=0.9687


In [99]:
print(df.dtypes)

power_factor_avg    float64
current_l1          float64
current_l2          float64
current_l3          float64
volts_l1_n          float64
volts_l2_n          float64
volts_l3_n          float64
power_total         float64
hour                  int32
dayofweek             int32
month                 int32
is_weekend            int64
lag_1               float64
lag_4               float64
lag_96              float64
rolling_mean_4      float64
rolling_std_4       float64
dtype: object
